# Query 4: Top 10 Most Expensive Trips
**Type:** Sorting and Ranking  
**Problem Statement:** Find the 10 most expensive trips by fare amount. Demonstrates how Catalyst optimizes top-N queries using TakeOrderedAndProject (min-heap) instead of a full global sort.

In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

spark = SparkSession.builder \
    .appName('Q4_Sorting') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.driver.memory', '2g') \
    .config('spark.executor.memory', '2g') \
    .config('spark.eventLog.enabled', 'false') \
    .config('spark.ui.enabled', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

26/04/25 19:54:23 WARN Utils: Your hostname, mariam-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/25 19:54:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 19:54:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1


In [2]:
df = spark.read.option('header', 'true').option('inferSchema', 'true') \
          .csv('../data/yellow_tripdata_2015-01.csv') \
          .sample(fraction=0.2, seed=42)

df = df.withColumnRenamed('tpep_pickup_datetime',  'pickup_datetime') \
       .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime') \
       .withColumnRenamed('fare_amount',            'fare') \
       .withColumnRenamed('passenger_count',        'passengers') \
       .withColumnRenamed('trip_distance',          'distance') \
       .withColumnRenamed('total_amount',           'total')

df = df.withColumn('fare',       F.col('fare').cast(DoubleType())) \
       .withColumn('total',      F.col('total').cast(DoubleType())) \
       .withColumn('distance',   F.col('distance').cast(DoubleType())) \
       .withColumn('passengers', F.col('passengers').cast(IntegerType())) \
       .withColumn('tip_amount', F.col('tip_amount').cast(DoubleType())) \
       .cache()

df.createOrReplaceTempView('trips')
rdd = df.rdd
print('Total rows (20% sample):', df.count())

Total rows (20% sample): 2233826


## RDD Implementation

In [3]:
start = time.time()

# RDD must sort entire dataset then take top 10 — no min-heap optimization
result_rdd = (
    rdd
    .filter(lambda r: r['fare'] is not None)
    .sortBy(lambda r: -r['fare'])
    .take(10)
)

rdd_time = time.time() - start
print(f'RDD | Time: {rdd_time:.2f}s')
print('Top 10 (RDD):')
for row in result_rdd:
    print(f"  fare=${row['fare']:.2f}  distance={row['distance']}mi  passengers={row['passengers']}")

RDD | Time: 35.95s
Top 10 (RDD):
  fare=$999.99  distance=0.0mi  passengers=1
  fare=$900.00  distance=0.0mi  passengers=1
  fare=$900.00  distance=0.0mi  passengers=1
  fare=$900.00  distance=0.0mi  passengers=1
  fare=$900.00  distance=0.0mi  passengers=1
  fare=$821.00  distance=0.0mi  passengers=1
  fare=$780.00  distance=0.0mi  passengers=4
  fare=$780.00  distance=6.8mi  passengers=1
  fare=$750.01  distance=0.2mi  passengers=2
  fare=$672.85  distance=0.0mi  passengers=1


## DataFrame Implementation

In [4]:
start = time.time()

# Catalyst uses TakeOrderedAndProject – a min-heap of 10 rows,
# much faster than a full global sort.
result_df = (
    df.select('pickup_datetime', 'dropoff_datetime',
              'passengers', 'distance', 'fare',
              'tip_amount', 'total', 'payment_type')
      .orderBy(F.desc('fare'))
      .limit(10)
)

print('--- Q4 DataFrame explain(True) ---')
result_df.explain(True)
result_df.cache()
df_time = time.time() - start
print(f'DataFrame | Time: {df_time:.2f}s')
result_df.show()
result_df.unpersist()

--- Q4 DataFrame explain(True) ---
== Parsed Logical Plan ==
GlobalLimit 10
+- LocalLimit 10
   +- Sort [fare#176 DESC NULLS LAST], true
      +- Project [pickup_datetime#55, dropoff_datetime#76, passengers#236, distance#216, fare#176, tip_amount#256, total#196, payment_type#28]
         +- Project [VendorID#17, pickup_datetime#55, dropoff_datetime#76, passengers#236, distance#216, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#176, extra#30, mta_tax#31, cast(tip_amount#32 as double) AS tip_amount#256, tolls_amount#33, improvement_surcharge#34, total#196]
            +- Project [VendorID#17, pickup_datetime#55, dropoff_datetime#76, cast(passengers#116 as int) AS passengers#236, distance#216, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#176, extra#30, mta_tax#31, tip_amount#32, tolls_amount#33,

+-------------------+-------------------+----------+--------+------+----------+-------+------------+
|    pickup_datetime|   dropoff_datetime|passengers|distance|  fare|tip_amount|  total|payment_type|
+-------------------+-------------------+----------+--------+------+----------+-------+------------+
|2015-01-16 14:48:00|2015-01-16 14:48:00|         1|     0.0|999.99|       0.0|1099.99|         1.0|
|2015-01-28 08:52:49|2015-01-28 08:53:18|         1|     0.0| 900.0|       0.0|  900.3|         1.0|
|2015-01-09 16:14:41|2015-01-09 16:15:11|         1|     0.0| 900.0|       0.0|  900.3|         1.0|
|2015-01-05 12:27:02|2015-01-05 12:27:28|         1|     0.0| 900.0|       0.0|  900.3|         1.0|
|2015-01-08 06:06:57|2015-01-08 06:07:16|         1|     0.0| 900.0|       0.0|  900.3|         1.0|
|2015-01-30 12:10:00|2015-01-30 12:10:00|         1|     0.0| 821.0|       0.0|1820.99|         1.0|
|2015-01-24 12:43:36|2015-01-24 12:44:35|         4|     0.0| 780.0|       0.0|  780.3|    

DataFrame[pickup_datetime: timestamp, dropoff_datetime: string, passengers: int, distance: double, fare: double, tip_amount: double, total: double, payment_type: double]

## Spark SQL Implementation

In [5]:
start = time.time()

result_sql = spark.sql("""
    SELECT   pickup_datetime, dropoff_datetime,
             passengers, distance, fare,
             tip_amount, total, payment_type
    FROM     trips
    ORDER BY fare DESC
    LIMIT    10
""")

print('--- Q4 Spark SQL explain(True) ---')
result_sql.explain(True)
result_sql.cache()
sql_time = time.time() - start
print(f'SQL | Time: {sql_time:.2f}s')
result_sql.show()
result_sql.unpersist()

--- Q4 Spark SQL explain(True) ---
== Parsed Logical Plan ==
'GlobalLimit 10
+- 'LocalLimit 10
   +- 'Sort ['fare DESC NULLS LAST], true
      +- 'Project ['pickup_datetime, 'dropoff_datetime, 'passengers, 'distance, 'fare, 'tip_amount, 'total, 'payment_type]
         +- 'UnresolvedRelation [trips], [], false

== Analyzed Logical Plan ==
pickup_datetime: timestamp, dropoff_datetime: string, passengers: int, distance: double, fare: double, tip_amount: double, total: double, payment_type: double
GlobalLimit 10
+- LocalLimit 10
   +- Sort [fare#176 DESC NULLS LAST], true
      +- Project [pickup_datetime#55, dropoff_datetime#76, passengers#236, distance#216, fare#176, tip_amount#256, total#196, payment_type#28]
         +- SubqueryAlias trips
            +- View (`trips`, [VendorID#17,pickup_datetime#55,dropoff_datetime#76,passengers#236,distance#216,pickup_longitude#22,pickup_latitude#23,RateCodeID#24,store_and_fwd_flag#25,dropoff_longitude#26,dropoff_latitude#27,payment_type#28,fare#176

+-------------------+-------------------+----------+--------+------+----------+-------+------------+
|    pickup_datetime|   dropoff_datetime|passengers|distance|  fare|tip_amount|  total|payment_type|
+-------------------+-------------------+----------+--------+------+----------+-------+------------+
|2015-01-16 14:48:00|2015-01-16 14:48:00|         1|     0.0|999.99|       0.0|1099.99|         1.0|
|2015-01-28 08:52:49|2015-01-28 08:53:18|         1|     0.0| 900.0|       0.0|  900.3|         1.0|
|2015-01-09 16:14:41|2015-01-09 16:15:11|         1|     0.0| 900.0|       0.0|  900.3|         1.0|
|2015-01-05 12:27:02|2015-01-05 12:27:28|         1|     0.0| 900.0|       0.0|  900.3|         1.0|
|2015-01-08 06:06:57|2015-01-08 06:07:16|         1|     0.0| 900.0|       0.0|  900.3|         1.0|
|2015-01-30 12:10:00|2015-01-30 12:10:00|         1|     0.0| 821.0|       0.0|1820.99|         1.0|
|2015-01-24 12:43:36|2015-01-24 12:44:35|         4|     0.0| 780.0|       0.0|  780.3|    

DataFrame[pickup_datetime: timestamp, dropoff_datetime: string, passengers: int, distance: double, fare: double, tip_amount: double, total: double, payment_type: double]

## Performance Comparison

In [ ]:
print('='*65)
row1 = f'{"Metric":<25} {"RDD":>12} {"DataFrame":>12} {"SQL":>12}'
row2 = f'{"Execution Time":<25} {rdd_time:>11.2f}s {df_time:>11.2f}s {sql_time:>11.2f}s'
row3 = f'{"Sort Strategy":<25} {"Full Sort":>12} {"Min-Heap":>12} {"Min-Heap":>12}'
row4 = f'{"Optimizer":<25} {"None":>12} {"Catalyst":>12} {"Catalyst":>12}'
print(row1)
print('-'*65)
print(row2)
print(row3)
print(row4)
print('='*65)
print()
print('KEY INSIGHT:')
print('RDD sortBy sorts ALL data then takes 10 — very expensive.')
print('Catalyst uses TakeOrderedAndProject (min-heap of size 10).')
print('No full sort needed — DataFrame/SQL are dramatically faster.')